In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import logging

In [9]:
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

In [10]:
# Base URL for paginated comments
BASE_URL = "https://www.pakistanifashionreviews.com/comments?page={}"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

In [11]:
def parse_comment(block):
    """
    Given a BeautifulSoup tag for one comment entry,
    extract brand, comment text, date, platform, language.
    """
    brand     = block.select_one(".brand-name").get_text(strip=True)
    comment   = block.select_one(".comment-text").get_text(strip=True)
    raw_date  = block.select_one(".comment-date").get_text(strip=True)
    # assuming dates like "February 21, 2024"
    date      = datetime.strptime(raw_date, "%B %d, %Y").date().isoformat()
    platform  = block.select_one(".comment-platform").get_text(strip=True)
    language  = block.select_one(".comment-language").get_text(strip=True)
    return {
        "brand":    brand,
        "comment":  comment,
        "date":     date,
        "platform": platform,
        "language": language
    }

In [12]:
def scrape_page(page_num):
    """
    Fetch one page of comments and return a list of dicts.
    """
    url = BASE_URL.format(page_num)
    resp = requests.get(url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    entries = soup.select("div.comment-entry")
    return [parse_comment(e) for e in entries]

In [13]:
def scrape_all(pages=10, pause=1.0):
    """
    Iterate through pages, collect all comment records.
    Stops early if a page returns no entries.
    """
    all_records = []
    for p in range(1, pages+1):
        logging.info(f"Scraping page {p}")
        records = scrape_page(p)
        if not records:
            logging.info("No more comments found; exiting.")
            break
        all_records.extend(records)
        time.sleep(pause)
    return pd.DataFrame(all_records)

In [ ]:
if __name__ == "__main__":
    df = scrape_all(pages=20, pause=0.5)
    df.to_csv("pakistani_fashion_comments.csv", index=False)
    logging.info(f"Saved {len(df)} comments.")